# Train the MLBB SCREEN detector (free, in Google Colab)

Makes `mlbb_screen_detector.tflite` + `screen_labels.txt`: a YOLO model that finds, on the **whole game screen**:
heroes (yours / ally / enemy), minions (ally / enemy), every jungle monster BY NAME, turrets (plated / normal / base) for both teams, bushes and walls.

**Jungle monsters (class names):** `lord` (human-like giant holding a trident), `turtle` (giant turtle), `molten_fiend` (the ORANGE buff, a fire monster - the wiki name; earlier called fiery_hound), `thunder_fenrir` (the PURPLE buff, a purple monster with a cub beside it - box the cub as thunder_fenrir too), `lava_golem` (a human-like rock), `horned_lizard` (a lizard), `fire_beetle`, `lithowanderer`, `crab`, and the terrain feature `cyclone_eye` (a blink pad, not a monster). There is no "blue buff" / "red buff" any more.
`jungle_creep` = a small monster the app could not name: **rename these boxes in Roboflow** to the right monster (the app names the lord, turtle, both buffs and the horned lizard itself). Box the Fire Beetle's little Crammer as fire_beetle.

**Turrets:** `tower_enemy_plated` / `tower_ally_plated` = outer turret with the gold energy shield (yellow HP bar, until 5:00); `tower_enemy` / `tower_ally` = without it; `base_enemy` / `base_ally` = the base (crystal). The app pre-draws enemy turrets only - add allied turrets and the bases in Roboflow.

**Where the pictures come from:** the app saves them while you play (Learn page → *Collect detector training pictures*),
with boxes already drawn from its colour vision, in `Download/MLBB-Research/screen_dataset`.

**Two ways:**
- **Quick:** zip the `screen_dataset` folder (file manager → compress) and upload the zip below. The model then learns what the app's colour vision already sees - mostly useful to also spot things WITHOUT an HP bar (a monster nobody has hit yet).
- **Best:** upload the folder to a Roboflow project (type *Object Detection*), fix / add boxes there, generate a version, and use the Roboflow cell instead. Corrected boxes are what make it really better.

*Runtime → Change runtime type → T4 GPU.*


**Retraining keeps what it learned:** each run continues from your last detector (`*_detector_best.pt`, saved in My Drive/MLBB-Research) and uses every picture zip in that folder. Keep the zips and that file in Drive.

In [ ]:
# ---- Started by the app (Learn page > Train)? Then this runs on Kaggle, unattended. ----
# It makes the notebook behave as in Colab: the files the app uploaded appear in "My Drive/MLBB-Research",
# downloads go to Kaggle's output (the app fetches them), and nothing waits for typing.
# In Colab this cell does nothing.
import os, sys, glob, shutil, types, zipfile, importlib.machinery
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.isdir("/kaggle/working")
def roboflow_key():       # Kaggle: the key from roboflow_key.txt (no typing); "" = none
    f = os.path.join("/content/drive/MyDrive/MLBB-Research", "roboflow_key.txt")
    return open(f).read().strip() if os.path.exists(f) else ""
if ON_KAGGLE:
    DRIVE = "/content/drive/MyDrive/MLBB-Research"
    os.makedirs(DRIVE, exist_ok=True)
    # Kaggle UNZIPS uploaded zips: a folder holding images/ + labels/ was a picture zip. Zip it again
    # under its own name (the notebook reads mlbb-*-dataset-*.zip) instead of copying every picture.
    # the uploads this run needs (the app writes their names here). One that is not attached yet
    # (Kaggle was still processing it) is waited for, then fetched directly with kagglehub.
    APP_SOURCES = []
    import time
    def _attached(ref):
        slug = ref.split("/")[-1]
        return any(slug in p.split("/") for p in glob.glob("/kaggle/input/**/", recursive=True))
    ROOTS = ["/kaggle/input"]
    for _ in range(12):
        missing = [r for r in APP_SOURCES if not _attached(r)]
        if not missing: break
        print("waiting for", missing); time.sleep(15)
    for ref in [r for r in APP_SOURCES if not _attached(r)]:
        try:
            import kagglehub
            ROOTS.append(kagglehub.dataset_download(ref)); print("fetched", ref, "directly")
        except Exception as e:
            print("could not fetch", ref, ":", e)
    zipped_roots = []
    for img_dir in sorted(set(os.path.dirname(p) for R in ROOTS for p in glob.glob(R + "/**/images/*", recursive=True))):
        root = os.path.dirname(os.path.dirname(img_dir))          # the folder above "<name>/images"
        if any(root == r or root.startswith(r + "/") for r in zipped_roots): continue
        parts = [x for x in img_dir.split("/") if x.startswith("mlbb-minimap-dataset") or x.startswith("mlbb-screen-dataset")]
        first = os.listdir(img_dir)[0] if os.listdir(img_dir) else ""
        kind = "minimap" if first.startswith("mm_") else "screen"
        name = parts[0] if parts else f"mlbb-{kind}-dataset-{len(zipped_roots)}"
        if not name.startswith("mlbb-"): name = "mlbb-" + name
        out = os.path.join(DRIVE, name + ".zip")
        with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
            for f in glob.glob(root + "/**/*", recursive=True):
                if os.path.isfile(f): z.write(f, os.path.relpath(f, root))
        zipped_roots.append(root)
        print("pictures from the app:", name + ".zip", len(os.listdir(img_dir)), "pictures")
    for f in [f for R in ROOTS for f in glob.glob(R + "/**/*", recursive=True)]:
        if not os.path.isfile(f) or any(f.startswith(r + "/") for r in zipped_roots): continue
        if "/gemma" in f.lower(): continue          # an attached model (Gemma), not a file from the app
        dst = os.path.join(DRIVE, os.path.basename(f))
        if os.path.exists(dst): continue
        # small files are copied (the notebook may overwrite them), big ones linked
        if os.path.getsize(f) < 50_000_000: shutil.copy(f, dst)
        else: os.symlink(f, dst)
    # the last tactical networks (Kaggle unzipped mlbb_policies.zip): back where the notebook looks for them,
    # so the training CONTINUES from them instead of starting over
    for f in glob.glob(os.path.join(DRIVE, "*.keras")):
        os.makedirs(os.path.join(DRIVE, "policy_models"), exist_ok=True)
        shutil.move(f, os.path.join(DRIVE, "policy_models", os.path.basename(f)))
    print("Files from the app:", sorted(os.listdir(DRIVE)))
    if not os.listdir(DRIVE):
        raise SystemExit("No files from the app arrived on Kaggle (the upload may still have been processing): "
                         "tap Train again in the app in a few minutes")
    KAGGLE_OUT = "/kaggle/working"
    def _download(p):
        shutil.copy(p, KAGGLE_OUT); print("result:", os.path.basename(p))
    # Libraries also reach for other parts of google.colab (e.g. the converter imports its "output"):
    # anything asked for exists and does nothing
    class _Anything:
        def __getattr__(self, n): return _Anything()
        def __call__(self, *a, **k): return _Anything()
        def __bool__(self): return False
        def __iter__(self): return iter(())
    class _ColabModule(types.ModuleType):
        def __getattr__(self, n):
            if n.startswith("__"): raise AttributeError(n)
            return _Anything()
    _colab = _ColabModule("google.colab"); _colab.__path__ = []
    _colab.__spec__ = importlib.machinery.ModuleSpec("google.colab", None, is_package=True)   # (libraries look it up)
    _colab.files = types.SimpleNamespace(download=_download, upload=lambda: {})
    _colab.drive = types.SimpleNamespace(mount=lambda *a, **k: None)
    _colab.userdata = types.SimpleNamespace(get=lambda k: os.environ.get(k))
    import google; google.colab = _colab
    sys.modules["google.colab"] = _colab


In [ ]:
!pip install -q ultralytics roboflow

## 1. Settings

In [ ]:
EPOCHS = 100     # more = better but slower
IMGSZ  = 640     # whole screen: small things (minions, monsters far away) need 640
CLASSES = ["hero_ally", "hero_enemy", "hero_self", "minion_ally", "minion_enemy",
           "lord", "turtle", "molten_fiend", "thunder_fenrir", "lava_golem", "horned_lizard", "fire_beetle", "lithowanderer", "crab",
           "cyclone_eye", "jungle_creep",
           "tower_ally", "tower_ally_plated", "tower_enemy", "tower_enemy_plated", "base_ally", "base_enemy", "bush", "wall"]   # same order as the app
# pictures saved by older app versions: "creep" -> jungle_creep; their "buff" boxes cannot be named
# (orange or purple?) and are left out - rename them in Roboflow to keep them
OLD_NAMES = {"creep": "jungle_creep", "red_buff": "molten_fiend", "orange_buff": "molten_fiend", "blue_buff": "thunder_fenrir", "purple_buff": "thunder_fenrir"}

## 2a. Your pictures as a zip (quick way)

In [ ]:
import zipfile, glob, os, random, shutil, yaml
# Your files: taken from Google Drive (My Drive/MLBB-Research) when they are there - nothing to upload.
# If that folder has none, you pick them on this device instead. (Google asks once to allow Drive access.)
import glob, os
from google.colab import files
DRIVE_DIR = "/content/drive/MyDrive/MLBB-Research"
paths = []
try:
    from google.colab import drive
    drive.mount("/content/drive")
    for pat in ["mlbb-screen-dataset-*.zip", "screen_dataset*.zip"]:
        paths += sorted(glob.glob(f"{DRIVE_DIR}/{pat}"))
    paths = sorted(set(paths))
except Exception as e:
    print("Google Drive not used:", e)
if paths:
    print(f"Using {len(paths)} screen picture zips from Google Drive (My Drive/MLBB-Research):", ", ".join(os.path.basename(p) for p in paths))
else:
    print("No screen picture zips in My Drive/MLBB-Research - pick them on this device:")
    up = files.upload()
    paths = [os.path.abspath(k) for k in up]
shutil.rmtree("/content/raw", ignore_errors=True)
for k in paths:
    if k.endswith(".zip"): zipfile.ZipFile(k).extractall("/content/raw/" + os.path.basename(k)[:-4])
imgs = sorted(glob.glob("/content/raw/**/images/*.jpg", recursive=True) + glob.glob("/content/raw/**/images/*.png", recursive=True))
print(len(imgs), "pictures")

# the zip's classes.txt decides what each box number means; convert by name to CLASSES
cls_files = glob.glob("/content/raw/**/classes.txt", recursive=True)
theirs = [l.strip() for l in open(cls_files[0]) if l.strip()] if cls_files else CLASSES
theirs = [OLD_NAMES.get(n, n) for n in theirs]
remap = {i: CLASSES.index(n) for i, n in enumerate(theirs) if n in CLASSES}
if "buff" in theirs: print("note: old 'buff' boxes are left out (orange or purple buff?) - rename them in Roboflow to use them")

random.seed(0); random.shuffle(imgs)
counts = [0] * len(CLASSES)
shutil.rmtree("/content/sd", ignore_errors=True)
for part, lst in (("train", imgs[:int(len(imgs) * 0.85)]), ("valid", imgs[int(len(imgs) * 0.85):])):
    os.makedirs(f"/content/sd/{part}/images"); os.makedirs(f"/content/sd/{part}/labels")
    for im in lst:
        lb = im.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        rows = []
        if os.path.exists(lb):
            for line in open(lb):
                v = line.split()
                if len(v) == 5 and int(v[0]) in remap:
                    c = remap[int(v[0])]; counts[c] += 1; rows.append(" ".join([str(c)] + v[1:]))
        shutil.copy(im, f"/content/sd/{part}/images/")
        open(f"/content/sd/{part}/labels/" + os.path.basename(lb), "w").write("\n".join(rows) + ("\n" if rows else ""))
yaml.safe_dump({"train": "/content/sd/train/images", "val": "/content/sd/valid/images", "nc": len(CLASSES), "names": CLASSES},
               open("/content/screen.yaml", "w"))
DATA = "/content/screen.yaml"
print("boxes per class:", dict(zip(CLASSES, counts)))
for n, c in zip(CLASSES, counts):
    if c < 30: print(f"NOTE: only {c} '{n}' boxes - it will hardly learn this one yet (play / collect more, or draw them in Roboflow)")

## 2b. OR: your corrected pictures from Roboflow (best way)
Skip 2a and run this instead. Your class names in Roboflow must be the CLASSES above.

In [ ]:
# WORKSPACE = "your-workspace"; PROJECT = "your-project"; VERSION = 1
# from getpass import getpass
# from roboflow import Roboflow
# rf = Roboflow(api_key=getpass("Paste your Roboflow API key and press Enter: "))
# ds = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8")
# DATA = f"{ds.location}/data.yaml"
# print("Classes:", yaml.safe_load(open(DATA))["names"])

## 3. Train a small YOLOv8 model

In [ ]:
from ultralytics import YOLO
# CONTINUES from your last trained detector when it is in My Drive/MLBB-Research (screen_detector_best.pt):
# it keeps what it learned and gets better with the new pictures (old zips in Drive are used again too).
# The first time (no file yet) it starts from the general YOLO 'nano' model (small and fast, good for phones).
import os, shutil
PREV = "/content/drive/MyDrive/MLBB-Research/screen_detector_best.pt"
if os.path.exists(PREV):
    print("Continuing from your last trained detector:", PREV)
    model = YOLO(PREV)
else:
    print("No earlier detector in My Drive/MLBB-Research: starting fresh")
    model = YOLO("yolov8n.pt")
model.train(data=DATA, epochs=EPOCHS, imgsz=IMGSZ, batch=16, patience=25)
best_path = model.trainer.best
print("Best weights:", best_path)
# keep it for next time (the next run continues from it)
shutil.copy(best_path, "/content/screen_detector_best.pt")
try:
    if os.path.isdir("/content/drive/MyDrive"):
        os.makedirs("/content/drive/MyDrive/MLBB-Research", exist_ok=True)
        shutil.copy(best_path, PREV)
        print("Saved for next time: My Drive/MLBB-Research/screen_detector_best.pt")
except Exception as e:
    print("(could not save it to Drive:", e, ") - download screen_detector_best.pt and put it in My Drive/MLBB-Research")

## 4. Export to LiteRT (.tflite)

In [ ]:
best = YOLO(best_path)
try:
    out = best.export(format="litert", imgsz=IMGSZ)
except Exception as e:
    print("litert export not available here, using tflite:", e)
    out = best.export(format="tflite", imgsz=IMGSZ)
found = [str(out)] if out and str(out).endswith(".tflite") else []
found += sorted(glob.glob(os.path.dirname(str(best_path)) + "/**/*.tflite", recursive=True))
found = [f for f in found if os.path.exists(f)]
pick = next((f for f in found if "float32" in f), found[0])
shutil.copy(pick, "/content/mlbb_screen_detector.tflite")
names = best.names
if isinstance(names, dict): names = [names[k] for k in sorted(names)]
open("/content/screen_labels.txt", "w").write(",".join(names))
print("mlbb_screen_detector.tflite from", pick, "| classes:", ",".join(names))

## 5. Download

In [ ]:
import shutil, os
# A. Direct download (works best on a computer)
from google.colab import files
files.download("/content/mlbb_screen_detector.tflite"); files.download("/content/screen_labels.txt")
try:   # also keep a copy in My Drive/MLBB-Research when Google Drive is connected
    if os.path.isdir("/content/drive/MyDrive"):
        os.makedirs("/content/drive/MyDrive/MLBB-Research", exist_ok=True)
        shutil.copy("/content/mlbb_screen_detector.tflite", "/content/drive/MyDrive/MLBB-Research/")
        shutil.copy("/content/screen_labels.txt", "/content/drive/MyDrive/MLBB-Research/")
        print("Also saved to Google Drive > My Drive/MLBB-Research")
except Exception as e: print("(not copied to Drive:", e, ")")

In [ ]:
# B. Save to Google Drive (most reliable on a phone)
from google.colab import drive
drive.mount("/content/drive")
shutil.copy("/content/mlbb_screen_detector.tflite", "/content/drive/MyDrive/mlbb_screen_detector.tflite")
shutil.copy("/content/screen_labels.txt", "/content/drive/MyDrive/screen_labels.txt")
print("Saved to Google Drive > My Drive")

## 6. Put it in the app
Put `mlbb_screen_detector.tflite` and `screen_labels.txt` in **Download/MLBB-Research** on the phone. The app loads them by itself
(Model page shows *Screen detector: ready*). It then adds monsters nobody has hit yet, extra bushes / walls, and enemy towers whose
circle is not shown. Keep collecting and re-train now and then.

In [ ]:
# ---- Kaggle: every result the app downloads, in Kaggle's output ----
if ON_KAGGLE:
    for name in ['mlbb_screen_detector.tflite', 'screen_labels.txt', 'screen_detector_best.pt']:
        for src in (f"/content/{name}", f"{DRIVE}/{name}"):
            if os.path.isfile(src):
                shutil.copy(src, KAGGLE_OUT); break
    print("Results for the app:", sorted(os.listdir(KAGGLE_OUT)))
